# LeetCode 3548 — Equal Sum Grid Partition II bằng Python

- Link: https://leetcode.com/problems/equal-sum-grid-partition-ii/
- Độ khó: **Hard**
- Chủ đề: **Tổng tiền tố theo hàng/cột**, **bảng băm vị trí xuất hiện**, **phân tích tính liên thông** của hình chữ nhật khi bỏ một ô, kỹ thuật **chuyển vị** để dùng lại code

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def canPartitionGrid(self, grid: List[List[int]]) -> bool:
```

## 1. Mục tiêu học

1. Duyệt mọi **đường cắt ngang** bằng tổng tiền tố theo hàng; xử lý **đường cắt dọc** bằng cách **chuyển vị** lưới rồi gọi lại cùng hàm.
2. Đưa điều kiện "bỏ tối đa một ô để hai tổng bằng nhau" về: phần lớn hơn phải chứa một ô có giá trị **đúng bằng hiệu** hai tổng.
3. Phân tích khi nào bỏ một ô làm **mất liên thông**: hình chữ nhật ≥ 2×2 luôn còn liên thông; dải một hàng/một cột chỉ được bỏ **ô ở đầu mút**.
4. Kiểm tra "giá trị `d` có nằm trong các hàng `0..i` không" trong `O(1)` bằng **hàng xuất hiện đầu tiên / cuối cùng** của mỗi giá trị.
5. Kiểm chứng bằng vét cạn (BFS kiểm tra liên thông) trên lưới nhỏ.

## 2. Đề bài & ví dụ

Cho lưới `m × n` số nguyên dương. Hỏi có thể thực hiện **một** đường cắt ngang hoặc **một** đường cắt dọc sao cho:

- Hai phần đều khác rỗng.
- Tổng hai phần **bằng nhau**, hoặc có thể làm bằng nhau bằng cách **bỏ qua (discount) tối đa một ô** (ở một trong hai phần).
- Nếu bỏ một ô thì phần chứa ô đó vẫn phải **liên thông** (đi lên/xuống/trái/phải qua các ô còn lại).

**Ví dụ 1:** `[[1,4],[2,3]]` → `true` (cắt ngang: 5 = 5).

**Ví dụ 2:** `[[1,2],[3,4]]` → `true` (cắt dọc: 4 và 6; bỏ ô 2 bên phải ⇒ 4 = 4, phần phải còn ô 4 — liên thông).

**Ví dụ 3:** `[[1,2,4],[2,3,5]]` → `false` (cắt ngang: 7 và 10; bỏ ô 3 ở giữa hàng dưới làm tách thành `[2]` và `[5]`).

**Ví dụ 4:** `[[4,1,8],[3,2,6]]` → `false`.

**Ràng buộc:** `1 ≤ m, n ≤ 10^5`, `2 ≤ m·n ≤ 10^5`, `1 ≤ grid[i][j] ≤ 10^5`.

In [1]:
from typing import List
examples = [
    ([[1, 4], [2, 3]], True),
    ([[1, 2], [3, 4]], True),
    ([[1, 2, 4], [2, 3, 5]], False),
    ([[4, 1, 8], [3, 2, 6]], False),
]
for g, exp in examples:
    print(g, '-> kỳ vọng', exp)

[[1, 4], [2, 3]] -> kỳ vọng True
[[1, 2], [3, 4]] -> kỳ vọng True
[[1, 2, 4], [2, 3, 5]] -> kỳ vọng False
[[4, 1, 8], [3, 2, 6]] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Tổng theo hàng và tổng tiền tố

`rows[i] = sum(grid[i])`. Cắt ngang sau hàng `i` ⇒ phần trên có tổng `top = rows[0] + … + rows[i]`, phần dưới `total − top`.

In [2]:
g = [[1, 2, 4], [2, 3, 5]]
rows = [sum(r) for r in g]
total = sum(rows)
top = 0
for i in range(len(g) - 1):
    top += rows[i]
    print(f"cắt sau hàng {i}: trên = {top}, dưới = {total - top}")

cắt sau hàng 0: trên = 7, dưới = 10


### 3.2. Chuyển vị lưới

`list(zip(*g))` biến cột thành hàng. Cắt dọc trên `g` ⇔ cắt ngang trên lưới chuyển vị.

In [3]:
t = [list(c) for c in zip(*g)]
print('chuyển vị:', t)

chuyển vị: [[1, 2], [2, 3], [4, 5]]


### 3.3. Kiểm tra liên thông bằng BFS

Dùng để viết lời giải vét cạn: từ một ô, lan sang 4 hướng trong tập ô cho trước, xem có đi hết không.

In [4]:
def is_connected(cells):
    cells = set(cells)
    if not cells:
        return False
    start = next(iter(cells))
    seen, st = {start}, [start]
    while st:
        r, c = st.pop()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            x = (r + dr, c + dc)
            if x in cells and x not in seen:
                seen.add(x)
                st.append(x)
    return len(seen) == len(cells)

row_cells = [(1, 0), (1, 1), (1, 2)]
print('bỏ ô giữa của một hàng:', is_connected([(1, 0), (1, 2)]), '| bỏ ô đầu:', is_connected([(1, 1), (1, 2)]))
block = [(r, c) for r in range(2) for c in range(3)]
print('hình 2x3 bỏ ô (0,1):', is_connected([x for x in block if x != (0, 1)]))

bỏ ô giữa của một hàng: False | bỏ ô đầu: True
hình 2x3 bỏ ô (0,1): True


### 3.4. Hàng xuất hiện đầu tiên / cuối cùng của mỗi giá trị

`first[v]` = chỉ số hàng nhỏ nhất chứa `v`; `last[v]` = lớn nhất. Khi đó:

- các hàng `0..i` chứa `v` ⇔ `first[v] ≤ i`;
- các hàng `i+1..m−1` chứa `v` ⇔ `last[v] > i`.

In [5]:
first, last = {}, {}
for i, row in enumerate([[5, 1], [2, 5], [3, 4]]):
    for v in row:
        first.setdefault(v, i)
        last[v] = i
print('first =', first, '| last =', last)

first = {5: 0, 1: 0, 2: 1, 3: 2, 4: 2} | last = {5: 1, 1: 0, 2: 1, 3: 2, 4: 2}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với một đường cắt, gọi `S1 ≥ S2` là tổng phần lớn và phần nhỏ, `d = S1 − S2`.

- `d = 0` ⇒ xong.
- Bỏ một ô ở phần **nhỏ** chỉ làm chênh lệch lớn thêm (giá trị dương) ⇒ vô ích.
- Bỏ một ô giá trị `x` ở phần **lớn**: cần `S1 − x = S2` ⇔ `x = d`.

Vậy câu hỏi: **phần lớn có ô giá trị `d` mà bỏ đi vẫn liên thông không?**

Khi nào bỏ một ô làm mất liên thông? Phần là một hình chữ nhật `h × w`:

| hình dạng | ô được phép bỏ |
|---|---|
| `h ≥ 2` và `w ≥ 2` | **mọi** ô (luôn còn liên thông) |
| `h = 1` (một hàng), `w ≥ 2` | chỉ hai ô **đầu mút** của hàng |
| `w = 1` (một cột), `h ≥ 2` | chỉ hai ô đầu mút của cột |
| `1 × 1` | bỏ đi thì rỗng — không bao giờ hợp lệ (vì phần kia có tổng dương) |

Tính tay ví dụ 3, cắt ngang: phần dưới `[2,3,5]` là một hàng, `d = 3`; ô 3 ở giữa ⇒ không được. Cắt dọc (chuyển vị):

- sau cột 0: trái `1+2 = 3`, phải `2+4+3+5 = 14`, `d = 11` — không có ô 11.
- sau cột 1: trái `1+2+2+3 = 8`, phải `4+5 = 9`, `d = 1`; phần phải là cột `[4,5]` — không có 1.

⇒ `false`.

In [6]:
def check_rectangle_removals(h, w):
    cells = [(r, c) for r in range(h) for c in range(w)]
    ok = [x for x in cells if is_connected([y for y in cells if y != x])]
    return ok

for h, w in [(1, 4), (3, 1), (2, 2), (3, 4)]:
    ok = check_rectangle_removals(h, w)
    print(f"{h}x{w}: số ô được bỏ = {len(ok)}/{h*w}", ok if len(ok) < h * w else '(mọi ô)')

1x4: số ô được bỏ = 2/4 [(0, 0), (0, 3)]
3x1: số ô được bỏ = 2/3 [(0, 0), (2, 0)]
2x2: số ô được bỏ = 4/4 (mọi ô)
3x4: số ô được bỏ = 12/12 (mọi ô)


## 5. Cách 1 — Brute Force: thử mọi đường cắt, mọi ô bỏ, BFS kiểm tra liên thông

In [7]:
def can_partition_brute(grid):
    m, n = len(grid), len(grid[0])
    cuts = []
    for i in range(1, m):
        cuts.append(([(r, c) for r in range(i) for c in range(n)],
                     [(r, c) for r in range(i, m) for c in range(n)]))
    for j in range(1, n):
        cuts.append(([(r, c) for r in range(m) for c in range(j)],
                     [(r, c) for r in range(m) for c in range(j, n)]))
    for A, B in cuts:
        sa = sum(grid[r][c] for r, c in A)
        sb = sum(grid[r][c] for r, c in B)
        if sa == sb:
            return True
        for X, sx, sy in ((A, sa, sb), (B, sb, sa)):
            for cell in X:
                if sx - grid[cell[0]][cell[1]] == sy and is_connected([x for x in X if x != cell]):
                    return True
    return False

for g, exp in examples:
    print(can_partition_brute(g), '| kỳ vọng', exp)

True | kỳ vọng True
True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


**Độ phức tạp:** `(m + n)` đường cắt × `mn` ô × BFS `O(mn)` ⇒ `O((m + n)(mn)^2)` — chỉ dùng cho lưới nhỏ.

**Nút thắt:** (1) BFS để kiểm tra liên thông — thay bằng quy tắc hình dạng; (2) quét mọi ô để tìm giá trị `d` — thay bằng tra cứu băm.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề liên thông:** một hình chữ nhật `h × w` với `h, w ≥ 2` bỏ đi một ô bất kỳ vẫn liên thông (mọi ô còn lại nối được với "vành" của hình). Với dải `1 × w` (hoặc `h × 1`), bỏ ô giữa sẽ cắt đôi; chỉ bỏ được ô đầu mút.

**Thuật toán cho cắt ngang** (cắt dọc: chuyển vị rồi làm lại):

1. Tính `rows`, `total`, `first[v]`, `last[v]`.
2. Với mỗi `i` từ 0 tới `m − 2`: `top`, `bot`, `d = |top − bot|`.
3. Nếu `top > bot` (phần trên lớn, kích thước `(i+1) × n`):
   - `i + 1 ≥ 2` và `n ≥ 2` ⇒ đúng nếu `first[d] ≤ i`;
   - `i = 0` (một hàng), `n ≥ 2` ⇒ đúng nếu `grid[0][0] = d` hoặc `grid[0][n−1] = d`;
   - `n = 1` (một cột) ⇒ đúng nếu `grid[0][0] = d` hoặc `grid[i][0] = d`.
4. Đối xứng cho `bot > top` với `last[d] > i` và các ô đầu mút của phần dưới.

Mọi bước `O(1)` ⇒ tổng `O(mn)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi đường cắt, quét phần lớn với quy tắc liên thông (O(mn · (m + n)))

Bỏ BFS, dùng bổ đề liên thông; vẫn quét từng ô của phần lớn để tìm giá trị `d`.

In [8]:
def removable_cells(r0, r1, c0, c1):
    """Các ô có thể bỏ trong hình chữ nhật hàng [r0,r1], cột [c0,c1] mà vẫn liên thông."""
    h, w = r1 - r0 + 1, c1 - c0 + 1
    if h >= 2 and w >= 2:
        return [(r, c) for r in range(r0, r1 + 1) for c in range(c0, c1 + 1)]
    if h == 1 and w == 1:
        return []
    if h == 1:
        return [(r0, c0), (r0, c1)]
    return [(r0, c0), (r1, c0)]

def can_partition_scan(grid):
    m, n = len(grid), len(grid[0])
    rects = []
    for i in range(1, m):
        rects.append(((0, i - 1, 0, n - 1), (i, m - 1, 0, n - 1)))
    for j in range(1, n):
        rects.append(((0, m - 1, 0, j - 1), (0, m - 1, j, n - 1)))
    for A, B in rects:
        sa = sum(grid[r][c] for r in range(A[0], A[1] + 1) for c in range(A[2], A[3] + 1))
        sb = sum(grid[r][c] for r in range(B[0], B[1] + 1) for c in range(B[2], B[3] + 1))
        if sa == sb:
            return True
        big, d = (A, sa - sb) if sa > sb else (B, sb - sa)
        if any(grid[r][c] == d for r, c in removable_cells(*big)):
            return True
    return False

for g, exp in examples:
    print(can_partition_scan(g), '| kỳ vọng', exp)

True | kỳ vọng True
True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


**Độ phức tạp:** `O(mn · (m + n))` — với `m·n = 10^5` và `m + n` cỡ vài trăm đến `10^5` vẫn quá chậm.

### 7.2. Cách 3 — Duy trì bộ đếm giá trị khi dời đường cắt

Giữ `Counter` cho phần trên và phần dưới; dời đường cắt xuống một hàng thì chuyển các giá trị của hàng đó từ "dưới" sang "trên". Tra `cnt[d] > 0` trong `O(1)`.

In [9]:
from collections import Counter

def horiz_counter(g):
    m, n = len(g), len(g[0])
    top_cnt, bot_cnt = Counter(), Counter(v for row in g for v in row)
    rows = [sum(r) for r in g]
    top, total = 0, sum(rows)
    for i in range(m - 1):
        for v in g[i]:
            top_cnt[v] += 1
            bot_cnt[v] -= 1
        top += rows[i]
        bot = total - top
        if top == bot:
            return True
        if top > bot:
            d = top - bot
            if i + 1 >= 2 and n >= 2:
                if top_cnt[d] > 0:
                    return True
            elif n >= 2:
                if d in (g[0][0], g[0][n - 1]):
                    return True
            elif d in (g[0][0], g[i][0]):
                return True
        else:
            d = bot - top
            if m - i - 1 >= 2 and n >= 2:
                if bot_cnt[d] > 0:
                    return True
            elif n >= 2:
                if d in (g[m - 1][0], g[m - 1][n - 1]):
                    return True
            elif d in (g[i + 1][0], g[m - 1][0]):
                return True
    return False

def can_partition_counter(grid):
    return horiz_counter(grid) or horiz_counter([list(c) for c in zip(*grid)])

for g, exp in examples:
    print(can_partition_counter(g), '| kỳ vọng', exp)

True | kỳ vọng True
True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


**Độ phức tạp:** mỗi ô được chuyển một lần ⇒ `O(mn)` cho mỗi chiều.

### 7.3. Cách 4 — `first` / `last` thay cho bộ đếm (lời giải chính)

Không cần cập nhật bộ đếm: "phần trên (hàng `0..i`) chứa `d`" ⇔ `first[d] ≤ i`; "phần dưới chứa `d`" ⇔ `last[d] > i`.

In [10]:
def horiz_firstlast(g):
    m, n = len(g), len(g[0])
    rows = [sum(r) for r in g]
    total = sum(rows)
    first, last = {}, {}
    for i, row in enumerate(g):
        for v in row:
            if v not in first:
                first[v] = i
            last[v] = i
    top = 0
    for i in range(m - 1):
        top += rows[i]
        bot = total - top
        if top == bot:
            return True
        if top > bot:
            d = top - bot
            if i >= 1 and n >= 2:
                if first.get(d, m) <= i:
                    return True
            elif n >= 2:
                if g[0][0] == d or g[0][n - 1] == d:
                    return True
            elif g[0][0] == d or g[i][0] == d:
                return True
        else:
            d = bot - top
            if m - i - 1 >= 2 and n >= 2:
                if last.get(d, -1) > i:
                    return True
            elif n >= 2:
                if g[m - 1][0] == d or g[m - 1][n - 1] == d:
                    return True
            elif g[i + 1][0] == d or g[m - 1][0] == d:
                return True
    return False

def can_partition_firstlast(grid):
    return horiz_firstlast(grid) or horiz_firstlast([list(c) for c in zip(*grid)])

for g, exp in examples:
    print(can_partition_firstlast(g), '| kỳ vọng', exp)

True | kỳ vọng True
True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 2 và ví dụ 3: in từng đường cắt (ngang trên lưới gốc, rồi "ngang" trên lưới chuyển vị = dọc), hình dạng phần lớn, `d`, và kết luận.

In [11]:
def trace(grid):
    for name, g in (('NGANG', grid), ('DỌC (chuyển vị)', [list(c) for c in zip(*grid)])):
        m, n = len(g), len(g[0])
        rows = [sum(r) for r in g]
        total = sum(rows)
        top = 0
        print(f"--- {name}: lưới {g}")
        for i in range(m - 1):
            top += rows[i]
            bot = total - top
            if top == bot:
                print(f"  cắt sau {i}: {top} = {bot} -> TRUE"); return True
            if top > bot:
                h, part, d = i + 1, 'trên', top - bot
                cand = sorted({v for row in g[:i + 1] for v in row}) if h >= 2 and n >= 2 else \
                       ([g[0][0], g[0][n - 1]] if n >= 2 else [g[0][0], g[i][0]])
            else:
                h, part, d = m - i - 1, 'dưới', bot - top
                cand = sorted({v for row in g[i + 1:] for v in row}) if h >= 2 and n >= 2 else \
                       ([g[m - 1][0], g[m - 1][n - 1]] if n >= 2 else [g[i + 1][0], g[m - 1][0]])
            ok = d in cand
            print(f"  cắt sau {i}: trên={top}, dưới={bot}, phần lớn = {part} ({h}x{n}), d={d}, "
                  f"giá trị được phép bỏ = {cand} -> {'TRUE' if ok else 'không'}")
            if ok:
                return True
    print('  => FALSE')
    return False

trace([[1, 2], [3, 4]])
print()
trace([[1, 2, 4], [2, 3, 5]])

--- NGANG: lưới [[1, 2], [3, 4]]
  cắt sau 0: trên=3, dưới=7, phần lớn = dưới (1x2), d=4, giá trị được phép bỏ = [3, 4] -> TRUE

--- NGANG: lưới [[1, 2, 4], [2, 3, 5]]
  cắt sau 0: trên=7, dưới=10, phần lớn = dưới (1x3), d=3, giá trị được phép bỏ = [2, 5] -> không
--- DỌC (chuyển vị): lưới [[1, 2], [2, 3], [4, 5]]
  cắt sau 0: trên=3, dưới=14, phần lớn = dưới (2x2), d=11, giá trị được phép bỏ = [2, 3, 4, 5] -> không
  cắt sau 1: trên=8, dưới=9, phần lớn = dưới (1x2), d=1, giá trị được phép bỏ = [4, 5] -> không
  => FALSE


False

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def canPartitionGrid(self, grid: List[List[int]]) -> bool:
        def horiz(g):
            m, n = len(g), len(g[0])
            rows = [sum(r) for r in g]
            total = sum(rows)
            first, last = {}, {}
            for i, row in enumerate(g):
                for v in row:
                    if v not in first:
                        first[v] = i
                    last[v] = i
            top = 0
            for i in range(m - 1):
                top += rows[i]
                bot = total - top
                if top == bot:
                    return True
                if top > bot:
                    d = top - bot
                    if i >= 1 and n >= 2:
                        if first.get(d, m) <= i:
                            return True
                    elif n >= 2:
                        if g[0][0] == d or g[0][n - 1] == d:
                            return True
                    elif g[0][0] == d or g[i][0] == d:
                        return True
                else:
                    d = bot - top
                    if m - i - 1 >= 2 and n >= 2:
                        if last.get(d, -1) > i:
                            return True
                    elif n >= 2:
                        if g[m - 1][0] == d or g[m - 1][n - 1] == d:
                            return True
                    elif g[i + 1][0] == d or g[m - 1][0] == d:
                        return True
            return False

        return horiz(grid) or horiz([list(c) for c in zip(*grid)])

for g, exp in examples:
    got = Solution().canPartitionGrid(g)
    print(got, 'OK' if got == exp else 'SAI')

True OK
True OK
False OK
False OK


## 10. Vì sao thuật toán đúng

1. **Chỉ bỏ ở phần lớn:** mọi giá trị dương; bỏ ở phần nhỏ làm hiệu tăng. Bỏ ở phần lớn giá trị `x` làm bằng nhau ⇔ `x = d`.
2. **Bổ đề liên thông:** trong hình chữ nhật `h × w` (`h, w ≥ 2`), bỏ ô `(r, c)`: mọi ô cùng hàng/cột với một ô kề vành vẫn đi vòng được; cụ thể mọi ô còn lại nối được với hàng hoặc cột không chứa ô bị bỏ, và các hàng/cột đó nối với nhau ⇒ liên thông. Trong dải `1 × w`, bỏ ô giữa tách hai bên; bỏ ô mút để lại dải liền. Dải `1 × 1` bỏ đi thành rỗng; trường hợp này không thể thỏa vì cần phần kia có tổng 0.
3. **Tra cứu đúng:** phần trên gồm các hàng `0..i` (đủ cả hàng) ⇒ chứa `d` ⇔ `first[d] ≤ i`; tương tự phần dưới với `last`.
4. **Cắt dọc = cắt ngang của lưới chuyển vị**, các hình dạng và quy tắc liên thông giữ nguyên (đổi vai hàng/cột).
5. Duyệt hết mọi đường cắt ⇒ không bỏ sót.

## 11. Phân tích Time / Space Complexity

Gọi `N = m·n ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn + BFS | `O((m + n) N^2)` | `O(N)` |
| 2. Quét phần lớn + quy tắc liên thông | `O((m + n) N)` | `O(N)` |
| 3. Bộ đếm cập nhật dần | `O(N)` | `O(N)` |
| 4. `first` / `last` | `O(N)` | `O(N)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [13]:
impls = {
    'scan': can_partition_scan,
    'counter': can_partition_counter,
    'firstlast': can_partition_firstlast,
    'submit': lambda g: Solution().canPartitionGrid(g),
}
edge = [
    [[5, 5]],                # 1x2: bằng nhau ngay
    [[3, 1, 2]],             # một hàng: [3] | [1, 2]
    [[1, 5, 2, 1]],          # một hàng, cần bỏ ô ở đầu mút
    [[2], [7], [3], [2]],    # một cột
    [[1, 3, 1]],             # [1] | [3, 1]: d = 3 ở đầu mút
    [[1], [1]],
    [[1], [2]],              # bỏ ô 2 thì phần dưới rỗng -> không hợp lệ
    [[1, 3, 1], [1, 1, 1]],  # hình 2 hàng
]
for g, exp in examples:
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
    assert can_partition_brute(g) == exp
for g in edge:
    b = can_partition_brute(g)
    for name, f in impls.items():
        assert f(g) == b, (name, g)
    print(g, '->', b)

[[5, 5]] -> True
[[3, 1, 2]] -> True
[[1, 5, 2, 1]] -> False
[[2], [7], [3], [2]] -> False
[[1, 3, 1]] -> True
[[1], [1]] -> True
[[1], [2]] -> False
[[1, 3, 1], [1, 1, 1]] -> False


Ở các ca biên, ta đối chiếu mọi cách với **vét cạn** thay vì tự đoán kết quả (dải một hàng/cột rất dễ tính nhầm). Ví dụ `[[1, 3, 1]]`: cắt `[1] | [3, 1]` có `d = 3`, ô 3 là **đầu mút** của `[3, 1]` nên bỏ được ⇒ `true`.

### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [14]:
import random
random.seed(3548)
cnt = 0
for _ in range(3000):
    m, n = random.randint(1, 4), random.randint(1, 4)
    if m * n < 2:
        continue
    g = [[random.randint(1, 6) for _ in range(n)] for _ in range(m)]
    exp = can_partition_brute(g)
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'lưới ngẫu nhiên.')

Khớp vét cạn trên 2799 lưới ngẫu nhiên.


### 12.3. Ca lớn (`m·n = 10^5`, nhiều hình dạng)

In [15]:
import time
random.seed(7)
shapes = [(316, 316), (1, 10**5), (10**5, 1), (2, 50000), (50000, 2)]
for m, n in shapes:
    g = [[random.randint(1, 10**5) for _ in range(n)] for _ in range(m)]
    t = time.time(); a = Solution().canPartitionGrid(g); t1 = time.time() - t
    t = time.time(); b = can_partition_counter(g); t2 = time.time() - t
    assert a == b
    print(f"{m}x{n}: {a} | first/last {t1:.2f}s | counter {t2:.2f}s")
# ca có đáp án True chắc chắn: hai nửa bằng nhau
g = [[1] * 1000 for _ in range(100)]
print('lưới toàn 1, 100x1000:', Solution().canPartitionGrid(g))

316x316: False | first/last 0.04s | counter 0.10s


1x100000: False | first/last 0.10s | counter 0.16s


100000x1: False | first/last 0.10s | counter 0.12s
2x50000: False | first/last 0.06s | counter 0.11s


50000x2: False | first/last 0.06s | counter 0.11s
lưới toàn 1, 100x1000: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép bỏ ô bất kỳ, quên điều kiện liên thông

In [16]:
def wrong_no_connectivity(grid):
    for g in (grid, [list(c) for c in zip(*grid)]):
        m = len(g)
        vals_top, rows = set(), [sum(r) for r in g]
        total, top = sum(rows), 0
        for i in range(m - 1):
            top += rows[i]
            bot = total - top
            topset = {v for row in g[:i + 1] for v in row}
            botset = {v for row in g[i + 1:] for v in row}
            if top == bot or (top - bot in topset) or (bot - top in botset):
                return True
    return False

print('sai :', wrong_no_connectivity([[1, 2, 4], [2, 3, 5]]))   # True
print('đúng:', Solution().canPartitionGrid([[1, 2, 4], [2, 3, 5]]))

sai : True
đúng: False


**Vì sao sai:** ví dụ 3 — bỏ ô giữa của một hàng làm phần đó bị tách.

### Lỗi 2 — Cấm hẳn việc bỏ ô trong dải một hàng (quên hai ô đầu mút)

In [17]:
def wrong_strip_never(grid):
    def horiz(g):
        m, n = len(g), len(g[0])
        rows = [sum(r) for r in g]
        total, top = sum(rows), 0
        for i in range(m - 1):
            top += rows[i]
            bot = total - top
            if top == bot:
                return True
            if top > bot and i >= 1 and n >= 2 and any(top - bot in row for row in g[:i + 1]):
                return True
            if bot > top and m - i - 1 >= 2 and n >= 2 and any(bot - top in row for row in g[i + 1:]):
                return True
        return False
    return horiz(grid) or horiz([list(c) for c in zip(*grid)])

g = [[1, 2], [3, 4]]
print('sai :', wrong_strip_never(g))
print('đúng:', Solution().canPartitionGrid(g))

sai : False
đúng: True


**Vì sao sai:** ví dụ 2 — cột phải `[2, 4]` là dải một cột; bỏ ô **đầu mút** 2 vẫn liên thông.

### Lỗi 3 — Tìm `d` ở phần **nhỏ** (hoặc cả hai phần)

In [18]:
def wrong_small_side(grid):
    for g in (grid, [list(c) for c in zip(*grid)]):
        m, n = len(g), len(g[0])
        rows = [sum(r) for r in g]
        total, top = sum(rows), 0
        for i in range(m - 1):
            top += rows[i]
            bot = total - top
            d = abs(top - bot)
            cells = [(r, c) for r in range(m) for c in range(n)]
            if top == bot:
                return True
            # SAI: chấp nhận ô ở BẤT KỲ phần nào, miễn bỏ đi vẫn liên thông
            for part in ([x for x in cells if x[0] <= i], [x for x in cells if x[0] > i]):
                for x in part:
                    if g[x[0]][x[1]] == d and is_connected([y for y in part if y != x]):
                        return True
    return False

random.seed(8)
found = None
for _ in range(5000):
    m, n = random.randint(1, 3), random.randint(1, 3)
    if m * n < 2:
        continue
    g = [[random.randint(1, 6) for _ in range(n)] for _ in range(m)]
    if wrong_small_side(g) != can_partition_brute(g):
        found = g
        break
print('phản ví dụ:', found, '| sai:', wrong_small_side(found), '| đúng:', Solution().canPartitionGrid(found))

phản ví dụ: [[4, 3], [6, 4]] | sai: True | đúng: False


**Vì sao sai:** bỏ ô ở phần nhỏ làm chênh lệch **tăng** thêm `d`, không bao giờ cân bằng được.

### Lỗi 4 — Chỉ xét cắt ngang, quên cắt dọc

In [19]:
random.seed(12)
found = None
for _ in range(5000):
    m, n = random.randint(2, 3), random.randint(2, 3)
    g = [[random.randint(1, 6) for _ in range(n)] for _ in range(m)]
    if not horiz_firstlast(g) and Solution().canPartitionGrid(g):
        found = g
        break
print('lưới:', found)
print('chỉ cắt ngang:', horiz_firstlast(found), '| đủ hai chiều:', Solution().canPartitionGrid(found))

lưới: [[6, 5, 2], [5, 1, 6], [5, 2, 4]]
chỉ cắt ngang: False | đủ hai chiều: True


**Vì sao sai:** có những lưới chỉ thỏa với đường cắt **dọc** (như lưới tìm được ở trên).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cắt lưới bằng một đường ngang/dọc" ⇒ tổng tiền tố theo hàng; **chuyển vị** để tái dùng code cho chiều còn lại.
- "Bỏ tối đa một phần tử để hai tổng bằng nhau" ⇒ cần phần tử bằng **đúng hiệu**, nằm ở phần lớn hơn.
- "Còn liên thông sau khi bỏ một ô" trên hình chữ nhật ⇒ chỉ dải 1 hàng/1 cột mới có vấn đề (ô giữa).
- "Phần có chứa giá trị `v` không" khi phần là tiền tố/hậu tố hàng ⇒ `first[v]`/`last[v]`.

## 15. Biến thể của bài

1. Bản I (LC 3546): không được bỏ ô ⇒ chỉ cần kiểm tra `top == bot`.
2. Được bỏ **tối đa k** ô ⇒ bài toán tổng con trên phần lớn + ràng buộc liên thông (khó hơn nhiều).
3. Cắt thành **ba** phần bằng hai đường song song ⇒ hai con trỏ trên tổng tiền tố.
4. Lưới có giá trị âm ⇒ có thể bỏ ô ở phần nhỏ (giá trị âm) — phải xét cả hai phần.

In [20]:
# Biến thể 1 (LC 3546): không bỏ ô
def can_partition_I(grid):
    for g in (grid, [list(c) for c in zip(*grid)]):
        rows = [sum(r) for r in g]
        total, top = sum(rows), 0
        for i in range(len(g) - 1):
            top += rows[i]
            if 2 * top == total:
                return True
    return False

print([can_partition_I(g) for g, _ in examples])

[True, False, False, False]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn + BFS | `O((m+n) N^2)` | `O(N)` | Oracle |
| 2 | Quét phần lớn + quy tắc liên thông | `O((m+n) N)` | `O(N)` | Lưới nhỏ |
| 3 | Bộ đếm dời dần | `O(N)` | `O(N)` | Dễ nghĩ |
| 4 | `first`/`last` | `O(N)` | `O(N)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1, 1, 3]]` → ?

**Bài 2 (tính tay):** `[[2, 1], [1, 2], [5, 1]]` → ?

**Bài 3 (điền TODO):** viết `removable_values(g, r0, r1)` trả về tập giá trị có thể bỏ của phần gồm các hàng `[r0, r1]`.

**Bài 4 (tự viết lại):** biến thể 4 — lưới có thể chứa số **âm** (và 0); sửa thuật toán và kiểm chứng bằng vét cạn.

**Bài 5 (mini challenge):** trả về **mô tả** một cách cắt hợp lệ (`('H', i)` hoặc `('V', j)`) cùng ô bị bỏ (nếu có), rồi kiểm tra lại.

In [21]:
def removable_values_todo(g, r0, r1):
    # TODO
    raise NotImplementedError

def can_partition_negative_todo(grid):
    # TODO
    raise NotImplementedError

def find_cut_todo(grid):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ có cắt dọc. (2) `[1,1] | [3]`: 2 vs 3, `d = 1`, phần lớn `[3]` là 1×1 — không bỏ được; `[1] | [1,3]`: 1 vs 4, `d = 3`, ô 3 là đầu mút. (3) `true`.

**Bài 2.** (1) Tổng hàng: 3, 3, 6 (tổng 12). (2) Cắt sau hàng 1: 6 vs 6. (3) `true`.

**Bài 3.** (1) Xác định `h = r1 − r0 + 1`, `w = n`. (2) Áp dụng bảng ở mục 4. (3) So với BFS.

**Bài 4.** (1) Có thể bỏ ô ở phần nhỏ nếu giá trị âm: cần `x = −d`. (2) Xét cả hai phần với giá trị mục tiêu tương ứng. (3) Vét cạn đã hỗ trợ sẵn mọi trường hợp.

**Bài 5.** (1) Như lời giải chính nhưng trả về thông tin thay vì `True`. (2) Với hình ≥2×2, tìm một ô có giá trị `d`. (3) Kiểm tra lại tổng và liên thông bằng BFS.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
a1 = Solution().canPartitionGrid([[1, 1, 3]])
a2 = Solution().canPartitionGrid([[2, 1], [1, 2], [5, 1]])
assert a1 is True and a1 == can_partition_brute([[1, 1, 3]])
assert a2 is True and a2 == can_partition_brute([[2, 1], [1, 2], [5, 1]])

def removable_values(g, r0, r1):
    n = len(g[0])
    return {g[r][c] for r, c in removable_cells(r0, r1, 0, n - 1)}

random.seed(9)
for _ in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    g = [[random.randint(1, 9) for _ in range(n)] for _ in range(m)]
    r0 = random.randrange(m); r1 = random.randrange(r0, m)
    cells = [(r, c) for r in range(r0, r1 + 1) for c in range(n)]
    exp = {g[r][c] for r, c in cells if is_connected([x for x in cells if x != (r, c)])}
    assert removable_values(g, r0, r1) == exp
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: True | Bài 2: True | Bài 3: đúng


In [23]:
# Bài 4: cho phép số âm và 0
def can_partition_negative(grid):
    m, n = len(grid), len(grid[0])
    rects = [((0, i - 1, 0, n - 1), (i, m - 1, 0, n - 1)) for i in range(1, m)] + \
            [((0, m - 1, 0, j - 1), (0, m - 1, j, n - 1)) for j in range(1, n)]
    for A, B in rects:
        sa = sum(grid[r][c] for r in range(A[0], A[1] + 1) for c in range(A[2], A[3] + 1))
        sb = sum(grid[r][c] for r in range(B[0], B[1] + 1) for c in range(B[2], B[3] + 1))
        if sa == sb:
            return True
        for X, sx, sy in ((A, sa, sb), (B, sb, sa)):
            need = sx - sy                       # cần bỏ ô giá trị need khỏi X
            if any(grid[r][c] == need for r, c in removable_cells(*X)):
                return True
    return False

random.seed(10)
for _ in range(2000):
    m, n = random.randint(1, 3), random.randint(1, 3)
    if m * n < 2:
        continue
    g = [[random.randint(-4, 4) for _ in range(n)] for _ in range(m)]
    # vét cạn: bỏ ô 1x1 thành rỗng -> tổng 0, coi là không hợp lệ (is_connected([]) = False)
    assert can_partition_negative(g) == can_partition_brute(g), g
print('Bài 4: khớp vét cạn với số âm')

Bài 4: khớp vét cạn với số âm


In [24]:
# Bài 5: trả về cách cắt
def find_cut(grid):
    m, n = len(grid), len(grid[0])
    for kind, A, B in [('H', (0, i - 1, 0, n - 1), (i, m - 1, 0, n - 1)) for i in range(1, m)] + \
                      [('V', (0, m - 1, 0, j - 1), (0, m - 1, j, n - 1)) for j in range(1, n)]:
        sa = sum(grid[r][c] for r in range(A[0], A[1] + 1) for c in range(A[2], A[3] + 1))
        sb = sum(grid[r][c] for r in range(B[0], B[1] + 1) for c in range(B[2], B[3] + 1))
        pos = A[1] if kind == 'H' else A[3]
        if sa == sb:
            return (kind, pos, None)
        big, d = (A, sa - sb) if sa > sb else (B, sb - sa)
        for r, c in removable_cells(*big):
            if grid[r][c] == d:
                return (kind, pos, (r, c))
    return None

for g, _ in examples:
    print(g, '->', find_cut(g))
    assert (find_cut(g) is not None) == Solution().canPartitionGrid(g)

[[1, 4], [2, 3]] -> ('H', 0, None)
[[1, 2], [3, 4]] -> ('H', 0, (1, 1))
[[1, 2, 4], [2, 3, 5]] -> None
[[4, 1, 8], [3, 2, 6]] -> None


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def horiz(g):
    rows, total, first[v], last[v]
    for i in 0..m-2: top += rows[i]; bot = total - top
        top == bot -> True
        phần lớn (h x n), d = |top - bot|:
            h >= 2 and n >= 2 -> tra first/last
            h == 1            -> hai ô mút của hàng
            n == 1            -> ô đầu và ô cuối của cột
return horiz(g) or horiz(transpose(g))
```

**Checklist:**

- [ ] Xét cả cắt ngang và cắt dọc (chuyển vị)?
- [ ] Chỉ bỏ ô ở phần **lớn**, giá trị đúng bằng hiệu?
- [ ] Dải một hàng/một cột: chỉ ô **đầu mút**?
- [ ] Phần `1 × 1` không được bỏ?
- [ ] Đối chiếu với vét cạn BFS trên lưới nhỏ?